**Summary**
- Vector DB
    - On RAM: FAISS
    - On Disk: Chroma
- Vector DB can be used as a reteriver.
- Reterival Strategies
    - MMR: This technique balances the relevance and the diversity for the reterived documents
    - MQR: Here another LLM is used to generate more user queries to get beter retervial from the system.
- Re-Ranking.
- Comparsion between BERT and Cross Encoder.


**Agenda for Today**
- Code for Re-Ranking
- Contenxt Compression
- Parent Document Retervial

In [4]:
!pip install -q langchain langchain_core langchain_openai langchain_community langchain_classic langchain_text_splitters faiss-cpu

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 84.8/84.8 kB 4.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.5/2.5 MB 39.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.0/1.0 MB 50.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 23.8/23.8 MB 55.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 64.7/64.7 kB 5.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 51.0/51.0 kB 4.8 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
google-colab 1.0.0 requires requests==2.32.4, but you have requests 2.32.5 which is incompatible.


In [5]:
import dotenv
import os
dotenv.load_dotenv("/content/.env")

True

In [6]:
import os
from langchain_openai import ChatOpenAI
llm = ChatOpenAI(
    openai_api_key=os.getenv("OPENROUTER_API_KEY"),
    openai_api_base=os.getenv("OPENROUTER_BASE_URL"),
    model_name="gpt-4o-mini-2024-07-18", # Or any model on OpenRouter
    default_headers={
        "HTTP-Referer": os.getenv("APP_URL"),
        "X-Title": os.getenv("APP_NAME"),
    }
)

In [7]:
from langchain_openai import OpenAIEmbeddings
embeddings = OpenAIEmbeddings(
    openai_api_key=os.getenv("OPENROUTER_API_KEY"),
    openai_api_base=os.getenv("OPENROUTER_BASE_URL"),
    model="text-embedding-3-small", # Or any model on OpenRouter
    default_headers={
        "HTTP-Referer": os.getenv("APP_URL"),
        "X-Title": os.getenv("APP_NAME"),
    })

**Re-Ranking**

- Load the text file
- Split it into chunks
- Setup vector DB
- Reteriver

In [9]:
from langchain_community.document_loaders import TextLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_community.vectorstores import FAISS

loader = TextLoader("/content/kb_modiji.txt")
documents = loader.load()

text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=1000,
    chunk_overlap=200)

chunks = text_splitter.split_documents(documents)
vs = FAISS.from_documents(documents=chunks, embedding=embeddings)
base_retriver = vs.as_retriever(search_kwargs={"k": 20})

**Re-Ranking Setup**

- In a general setup, if the chunks are small and we keep the value of k as small, we might miss out the entire context required to answer the query of the user.
- But on the other hand, keeping k value to be very large, bombards the llm with irrelevant information (chunk) which is again not a good technique.
- So here, re-ranker help her to find the most relevant chunks based on my query and then we provide it to the LLM to answer the query.

In [10]:
!pip install -q flashrank

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 17.1/17.1 MB 53.4 MB/s eta 0:00:00


In [11]:
from langchain_classic.retrievers import ContextualCompressionRetriever
from langchain_community.document_compressors.flashrank_rerank import FlashrankRerank


# this piece of code may be not required in upcoming version of FlashRanker
# as of now since this libaray is still in development and multiple updates are
# getting made, we have to use this forcefull variable allocation.
import langchain_community.document_compressors.flashrank_rerank as fr_module
from flashrank import Ranker, RerankRequest
fr_module.RerankRequest = RerankRequest
fr_module.Ranker = Ranker
FlashrankRerank.model_rebuild()


# initialise the compressor
compressor  = FlashrankRerank(top_n = 5)

# let's tie everything
compression_retriever = ContextualCompressionRetriever(
    base_retriever=base_retriver,
    base_compressor=compressor)


ms-marco-MultiBERT-L-12.zip: 100%|██████████| 98.7M/98.7M [00:01<00:00, 82.6MiB/s]


In [12]:
compression_retriever.invoke("When was he born?").relevance_score

[Document(metadata={'id': 3, 'relevance_score': np.float32(0.98845434), 'source': '/content/kb_modiji.txt'}, page_content="## Personal Life\n\n- Married to Jashodaben Chimanlal Modi (estranged marriage from youth)\n- Known for simple lifestyle and long working hours\n- Vegetarian and teetotaler\n- Practices yoga and meditation\n- Interests in photography and writing\n\n## Legacy and Impact\n\nNarendra Modi remains one of the most influential and polarizing figures in contemporary Indian politics. His tenure has been marked by significant policy initiatives, economic reforms, and a distinctive approach to governance and foreign policy. Supporters credit him with strong leadership, economic development, and enhanced global standing, while critics raise concerns about democratic values, inclusive growth, and social harmony.\n\n---\n\n*Note: This document presents a factual overview of Narendra Modi's history and career. He continues to serve as Prime Minister of India, and his legacy rema

In [13]:
# prompt
template = """
You MUST answer ONLY using the context below.
If the answer is not present, reply exactly:
"I don't know based on the provided context."

Context:
{context}

Question: {question}
"""

from langchain_core.prompts import ChatPromptTemplate
from langchain_core.runnables import RunnablePassthrough
from langchain_core.output_parsers import StrOutputParser
prompt = ChatPromptTemplate.from_template(template)

def format_doc(docs):
  return "\n\n".join([doc.page_content for doc in docs])

rag_chain = (
    {"context": compression_retriever| format_doc,
     "question": RunnablePassthrough()}
    |prompt
    |llm
    |StrOutputParser()
)

q = input("Enter the qustion about Narendra Modi ")
response = rag_chain.invoke(q)
print(response)

Enter the qustion about Narendra Modi His major initiatives
- Swachh Bharat Abhiyan (Clean India Mission)
- Make in India campaign
- Digital India initiative
- Jan Dhan Yojana (financial inclusion)
- Skill India program
- Pradhan Mantri Ujjwala Yojana (LPG connections)
- Goods and Services Tax (GST) implementation (2017)
- Demonetization (November 2016)


**Context Compression**

In [14]:
!pip install -qU llmlingua accelerate

In [16]:
#assume that document as been loaded, chuking is takecare and
# base retriver is ready

from llmlingua import PromptCompressor

llm_lingua = PromptCompressor(
    model_name = "microsoft/llmlingua-2-bert-base-multilingual-cased-meetingbank",
    use_llmlingua2= True,
    device_map = "cpu"
)

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

In [19]:
def context_compress(query):
  docs = base_retriver.invoke(query)

  context = "\n\n".join([doc.page_content for doc in docs])

  compressed_results = llm_lingua.compress_prompt(
      context,
      rate = 0.7,
      target_token = 800
  )

  return compressed_results['compressed_prompt']

In [20]:
template = """
You MUST answer ONLY using the context below.
If the answer is not present, reply exactly:
"I don't know based on the provided context."

Context:
{context}

Question: {question}
"""


from langchain_core.prompts import ChatPromptTemplate
from langchain_core.runnables import RunnablePassthrough
from langchain_core.output_parsers import StrOutputParser
prompt = ChatPromptTemplate.from_template(template)

rag_chain = (
    {"context":context_compress, "question": RunnablePassthrough()}
    |prompt
    |llm
    |StrOutputParser()
)

q = input("Enter the qustion about Narendra Modi ")
response = rag_chain.invoke(q)
print(response)

Enter the qustion about Narendra Modi Where was he born?


Token indices sequence length is longer than the specified maximum sequence length for this model (1762 > 512). Running this sequence through the model will result in indexing errors


He was born in Vadnagar, Gujarat.


**Parent Document Retrieval**

**Summary**
- Implemented Re-Raking
- Concept of Perplexity
- Contextual Compression
- Parent Document Retreival (Code Implimentation is pending)